# M3-B1 — Exploration des 3 sources Acerox

> Notebook d'**exploration rapide** destiné à alimenter `identification_sources.md`.
> L'objectif est d'observer les sources, pas de construire le pipeline d'ingestion.

**Auteur :** Fernando  
**Date :** 21 juillet 2026

## Règles

- Lecture et exploration uniquement : `info()`, `head()`, `describe()`, comptages.
- Pas de nettoyage définitif, de feature engineering ou de modèle ML.
- Les anomalies sont observées et documentées, mais pas corrigées dans ce notebook.
- Les risques RGPD et les questions ouvertes sont conservés pour la note d'identification.


In [18]:
from collections import Counter
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import display

DATA_DIR = Path("../data")

required_files = [
    "capteurs_iot.csv",
    "erp_export.json",
    "logs_machines.log",
]

missing_files = [
    filename
    for filename in required_files
    if not (DATA_DIR / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        "Fichiers manquants dans data/ : " + ", ".join(missing_files)
    )

print("Dossier de données :", DATA_DIR.resolve())
print("Les 3 sources principales sont disponibles.")


Dossier de données : /Users/fernando/Documents/devid/formation-ai/M3-B1-acerox-fernando/data
Les 3 sources principales sont disponibles.


## Source 1 — Capteurs IoT (`capteurs_iot.csv`)

Cette source contient les mesures physiques produites par les capteurs installés sur les lignes de production.


In [19]:
df_iot = pd.read_csv(DATA_DIR / "capteurs_iot.csv")

display(df_iot.head())

print("\nStructure de la source IoT :")
df_iot.info()

print("\nStatistiques descriptives :")
display(df_iot.describe(include="all").T)


,timestamp,site,line_id,sensor_id,temperature_c,vibration_mms,debit_uh
0,2026-04-14T19:21:43,Lyon,1,SLYO-L1-T01,77.92,5.539793,101.27
1,2026-04-27T02:47:12,Lyon,1,SLYO-L1-T01,70.58,3.361715,110.19
2,2026-04-13T18:18:50,Saint-Etienne,1,SSAI-L1-T01,62.37,4.019277,111.28
3,2026-04-05T10:34:03,Roubaix,2,SROU-L2-T01,66.17,4.922531,123.93
4,2026-04-20T10:18:07,Saint-Etienne,3,SSAI-L3-T01,55.56,1.643043,101.40



Structure de la source IoT :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51000 entries, 0 to 50999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      51000 non-null  object 
 1   site           51000 non-null  object 
 2   line_id        51000 non-null  int64  
 3   sensor_id      51000 non-null  object 
 4   temperature_c  51000 non-null  float64
 5   vibration_mms  50251 non-null  float64
 6   debit_uh       51000 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 2.7+ MB

Statistiques descriptives :


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
timestamp,51000,49505,2026-04-08T16:39:30,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
site,51000,3,Roubaix,20570,NaN,NaN,NaN,NaN,NaN,NaN,NaN
line_id,51000.0,NaN,NaN,NaN,2.005882,1.033472,1.0,1.0,2.0,3.0,4.0
sensor_id,51000,8,SLYO-L1-T01,10182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temperature_c,51000.0,NaN,NaN,NaN,73.717034,27.005577,26.47,60.2375,66.09,72.84,160.0
vibration_mms,50251.0,NaN,NaN,NaN,4.831502,2.685963,0.0,3.302777,4.187726,5.171534,12.0
debit_uh,51000.0,NaN,NaN,NaN,110.050188,11.832749,80.0,102.0,110.1,118.05,150.0


In [20]:
print("Volume :", df_iot.shape[0], "lignes x", df_iot.shape[1], "colonnes")
print(
    "Période :",
    df_iot["timestamp"].min(),
    "→",
    df_iot["timestamp"].max(),
)

print("\nRépartition par site :")
display(df_iot["site"].value_counts(dropna=False).rename("nombre_mesures"))

print("\nRépartition par ligne de production :")
display(df_iot["line_id"].value_counts(dropna=False).sort_index().rename("nombre_mesures"))

print("\nCardinalités :")
print("- Nombre de sites :", df_iot["site"].nunique(dropna=True))
print("- Nombre de lignes :", df_iot["line_id"].nunique(dropna=True))
print("- Nombre de capteurs :", df_iot["sensor_id"].nunique(dropna=True))

missing_iot = pd.DataFrame(
    {
        "valeurs_manquantes": df_iot.isna().sum(),
        "pourcentage": (df_iot.isna().mean() * 100).round(2),
    }
)
print("\nValeurs manquantes :")
display(missing_iot)

print("Doublons complets :", df_iot.duplicated().sum())
print(
    "Doublons sur timestamp + site + ligne + capteur :",
    df_iot.duplicated(
        subset=["timestamp", "site", "line_id", "sensor_id"]
    ).sum(),
)


Volume : 51000 lignes x 7 colonnes
Période : 2026-04-01T00:00:38 → 2026-04-29T23:57:53

Répartition par site :


site
Roubaix          20570
Saint-Etienne    20248
Lyon             10182
Name: nombre_mesures, dtype: int64


Répartition par ligne de production :


line_id
1    21980
2    11846
3    12068
4     5106
Name: nombre_mesures, dtype: int64


Cardinalités :
- Nombre de sites : 3
- Nombre de lignes : 4
- Nombre de capteurs : 8

Valeurs manquantes :


,valeurs_manquantes,pourcentage
timestamp,0,0.00
site,0,0.00
line_id,0,0.00
sensor_id,0,0.00
temperature_c,0,0.00
vibration_mms,749,1.47
debit_uh,0,0.00


Doublons complets : 1000
Doublons sur timestamp + site + ligne + capteur : 1073


## Observations — Capteurs IoT

**Volume :** 51 000 lignes et 7 colonnes.

**Période :** du 1er avril 2026 à 00:00:38 au 29 avril 2026 à 23:57:53, soit environ un mois de données.

**Fréquence annoncée :** une mesure par seconde.

**Qualité observée :**

* les colonnes principales sont globalement renseignées ;
* `vibration_mms` contient 749 valeurs manquantes, soit environ 1,47 % des lignes ;
* `timestamp` est initialement lu comme du texte ;
* 1 000 lignes sont entièrement dupliquées, soit environ 1,96 % du fichier ;
* la température maximale atteint 160 °C, alors que 75 % des valeurs sont inférieures à 72,84 °C : cette valeur extrême doit être vérifiée ;
* l’unité de `debit_uh` doit être confirmée ;
* la source contient trois sites : Lyon, Roubaix et Saint-Étienne.

**Risque RGPD :**
Aucune identité directe n’apparaît dans cette source. Cependant, le croisement de `site`, `line_id`, `sensor_id` et `timestamp` avec l’ERP, les logs ou un planning peut permettre d’associer une activité à une équipe ou à une personne.

**Pertinence métier :**
Cette source est prioritaire pour observer les conditions physiques précédant une non-conformité. La température, la vibration et le débit peuvent aider à détecter une dérive machine avant la production d’une pièce défectueuse.

**Questions pour Sébastien :**

1. Quelle est l’unité exacte de `debit_uh` ?
2. Pourquoi certaines mesures de vibration sont-elles absentes ?
3. Les 1 000 doublons proviennent-ils d’un rejeu d’export ou correspondent-ils à de vraies mesures répétées ?
4. Les températures proches de 160 °C sont-elles physiquement normales ?
5. La fréquence d’une mesure par seconde s’applique-t-elle à chaque capteur ?
6. Pourquoi le site de Lyon apparaît-il dans les capteurs alors qu’il n’apparaît pas dans l’ERP ?

## Source 2 — ERP (`erp_export.json`)

Cette source apporte le contexte des ordres de production : produit, site, ligne, dates, statut, opérateur et quantité.


In [21]:
with (DATA_DIR / "erp_export.json").open(encoding="utf-8") as file:
    orders = json.load(file)

if isinstance(orders, dict):
    print("Clés trouvées à la racine du JSON :", list(orders.keys()))

    # Certains exports enveloppent la liste dans une clé.
    list_values = [
        value for value in orders.values()
        if isinstance(value, list)
    ]

    if len(list_values) != 1:
        raise ValueError(
            "La structure JSON ne contient pas une liste unique identifiable."
        )

    orders = list_values[0]

df_erp = pd.DataFrame(orders)

display(df_erp.head())

print("\nStructure de la source ERP :")
df_erp.info()

print("\nStatistiques descriptives :")
display(df_erp.describe(include="all").T)


,ordre_id,produit_ref,site,line_id,date_lancement,date_fin_prevue,statut,ouvrier_id,quantite_kg
0,100000,ALU-T1-22,Roubaix,3,2026-04-01T22:21:08,2026-04-02T23:21:08,suspendu,EMP-5317,3221
1,100001,INOX-316-4,Saint-Etienne,1,2026-04-26T14:52:52,2026-04-28T15:52:52,termine,EMP-7240,4556
2,100002,ALU-T2-18,Saint-Etienne,3,2026-04-11T09:54:06,2026-04-12T16:54:06,suspendu,EMP-1939,1308
3,100003,ALU-T1-22,Roubaix,1,2026-04-20T22:33:08,2026-04-22T04:33:08,termine,EMP-3531,2968
4,100004,ALU-T2-25,Roubaix,4,2026-04-24T01:03:02,2026-04-25T21:03:02,termine,EMP-8778,3278



Structure de la source ERP :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ordre_id         2000 non-null   int64 
 1   produit_ref      2000 non-null   object
 2   site             2000 non-null   object
 3   line_id          2000 non-null   int64 
 4   date_lancement   2000 non-null   object
 5   date_fin_prevue  2000 non-null   object
 6   statut           2000 non-null   object
 7   ouvrier_id       1891 non-null   object
 8   quantite_kg      2000 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 140.8+ KB

Statistiques descriptives :


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ordre_id,2000.0,NaN,NaN,NaN,100999.5,577.494589,100000.0,100499.75,100999.5,101499.25,101999.0
produit_ref,2000,10,INOX-316-4,227,NaN,NaN,NaN,NaN,NaN,NaN,NaN
site,2000,2,Roubaix,1108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
line_id,2000.0,NaN,NaN,NaN,2.316,1.033769,1.0,1.0,2.0,3.0,4.0
date_lancement,2000,1999,2026-04-29T19:55:27,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_fin_prevue,2000,1999,2026-04-14T22:51:14,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
statut,2000,4,termine,1559,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ouvrier_id,1891,1689,EMP-4182,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantite_kg,2000.0,NaN,NaN,NaN,2528.689,1425.269007,51.0,1270.25,2534.0,3739.25,4999.0


In [22]:
print("Volume :", df_erp.shape[0], "lignes x", df_erp.shape[1], "colonnes")
print(
    "Période de lancement :",
    df_erp["date_lancement"].min(),
    "→",
    df_erp["date_lancement"].max(),
)

print("\nRépartition par site :")
display(df_erp["site"].value_counts(dropna=False).rename("nombre_ordres"))

print("\nRépartition par statut :")
display(df_erp["statut"].value_counts(dropna=False).rename("nombre_ordres"))

print("\nCardinalités :")
print("- Références produit :", df_erp["produit_ref"].nunique(dropna=True))
print("- Sites :", df_erp["site"].nunique(dropna=True))
print("- Lignes de production :", df_erp["line_id"].nunique(dropna=True))
print("- Ouvriers identifiés :", df_erp["ouvrier_id"].nunique(dropna=True))

missing_erp = pd.DataFrame(
    {
        "valeurs_manquantes": df_erp.isna().sum(),
        "pourcentage": (df_erp.isna().mean() * 100).round(2),
    }
)
print("\nValeurs manquantes :")
display(missing_erp)

print("Ordre_id dupliqués :", df_erp["ordre_id"].duplicated().sum())
print("Doublons complets :", df_erp.duplicated().sum())


Volume : 2000 lignes x 9 colonnes
Période de lancement : 2026-04-01T00:09:57 → 2026-04-29T23:20:33

Répartition par site :


site
Roubaix          1108
Saint-Etienne     892
Name: nombre_ordres, dtype: int64


Répartition par statut :


statut
termine     1559
en_cours     197
suspendu     139
annule       105
Name: nombre_ordres, dtype: int64


Cardinalités :
- Références produit : 10
- Sites : 2
- Lignes de production : 4
- Ouvriers identifiés : 1689

Valeurs manquantes :


,valeurs_manquantes,pourcentage
ordre_id,0,0.00
produit_ref,0,0.00
site,0,0.00
line_id,0,0.00
date_lancement,0,0.00
date_fin_prevue,0,0.00
statut,0,0.00
ouvrier_id,109,5.45
quantite_kg,0,0.00


Ordre_id dupliqués : 0
Doublons complets : 0


## Observations — ERP

**Volume :** 2 000 lignes et 9 colonnes.

**Période :** du 1er avril 2026 à 00:09:57 au 29 avril 2026 à 23:20:33.

**Fréquence annoncée :** export quotidien.

**Qualité observée :**

* chaque ligne semble représenter un ordre de production ;
* `ouvrier_id` contient 109 valeurs manquantes, soit environ 5,45 % des lignes ;
* les dates sont initialement lues comme du texte ;
* aucun doublon n’est observé sur `ordre_id` ;
* aucun doublon complet n’est observé ;
* quatre statuts apparaissent : `termine`, `en_cours`, `suspendu` et `annule` ;
* l’ERP contient uniquement les sites de Roubaix et Saint-Étienne, contrairement aux capteurs et aux logs qui contiennent également Lyon.

**Risque RGPD :**

* `ouvrier_id` est une donnée personnelle pseudonymisée ou indirectement identifiante ;
* son croisement avec les horaires, les logs machines, le site, la ligne ou un planning RH peut permettre de reconstruire l’activité d’un salarié ;
* il faut vérifier si cet identifiant est réellement nécessaire pour améliorer la prédiction des non-conformités.

**Pertinence métier :**

* l’ERP apporte le contexte de fabrication ;
* il permet d’associer un produit, un site, une ligne, une période, un statut et une quantité à un ordre de production ;
* il peut aider à expliquer dans quelles conditions une non-conformité apparaît ;
* il faudra toutefois identifier une clé ou une règle temporelle fiable pour le relier aux capteurs et aux logs.

**Questions pour Sébastien :**

1. Pourquoi certains ordres ne possèdent-ils pas d’`ouvrier_id` ?
2. Que signifient précisément les différents statuts ?
3. Quelle clé permet de relier un ordre ERP aux mesures IoT, aux logs et aux résultats qualité ?
4. Pourquoi le site de Lyon n’apparaît-il pas dans cet export ?
5. L’identité de l’ouvrier est-elle réellement nécessaire pour le modèle ?


## Source 3 — Logs machines (`logs_machines.log`)

Cette source texte contient des événements horodatés produits par les machines et les lignes de production.


In [23]:
log_path = DATA_DIR / "logs_machines.log"

with log_path.open(encoding="utf-8") as file:
    log_lines = [line.rstrip("\n") for line in file]

print(f"Nombre de lignes : {len(log_lines):,}")
print(f"Taille fichier : {log_path.stat().st_size / 1024:.1f} Ko")

print("\nPremière ligne :")
print(log_lines[0])

print("\nDernière ligne :")
print(log_lines[-1])

print("\nAperçu des 5 premières lignes :")
for line in log_lines[:5]:
    print(line)


Nombre de lignes : 30,000
Taille fichier : 1872.8 Ko

Première ligne :
[2026-04-01T00:00:16] Lyon LINE-1 INFO: shift_changed

Dernière ligne :
[2026-04-29T23:57:41] Roubaix LINE-2 INFO: machine_started

Aperçu des 5 premières lignes :
[2026-04-01T00:00:16] Lyon LINE-1 INFO: shift_changed
[2026-04-01T00:01:07] Saint-Etienne LINE-2 INFO: operator_login
[2026-04-01T00:01:34] Saint-Etienne LINE-3 ERROR: vibration_overlimit sensor=SSAI-L3-T01
[2026-04-01T00:04:18] Roubaix LINE-4 INFO: maintenance_completed
[2026-04-01T00:04:35] Lyon LINE-1 INFO: tooling_loaded


In [24]:
# Exploration légère du format des logs, sans construire de pipeline de parsing.
log_pattern = re.compile(
    r"^\[(?P<timestamp>[^\]]+)\]\s+"
    r"(?P<site>.+?)\s+"
    r"(?P<line>LINE-\d+)\s+"
    r"(?P<level>INFO|WARN|WARNING|ERROR):\s+"
    r"(?P<message>.+)$"
)

levels = Counter()
events = Counter()
sites = Counter()
production_lines = Counter()
malformed_examples = []

first_timestamp = None
last_timestamp = None

for line in log_lines:
    match = log_pattern.match(line)

    if not match:
        if len(malformed_examples) < 5:
            malformed_examples.append(line)
        continue

    data = match.groupdict()
    timestamp = data["timestamp"]
    message = data["message"]
    event_name = message.split()[0]

    if first_timestamp is None:
        first_timestamp = timestamp
    last_timestamp = timestamp

    levels[data["level"]] += 1
    events[event_name] += 1
    sites[data["site"]] += 1
    production_lines[data["line"]] += 1

print("Période observée :", first_timestamp, "→", last_timestamp)
print("Lignes reconnues :", sum(levels.values()))
print("Lignes non reconnues :", len(log_lines) - sum(levels.values()))

print("\nNiveaux de logs :")
display(
    pd.Series(levels, name="nombre_evenements")
    .sort_values(ascending=False)
    .to_frame()
)

print("\nÉvénements les plus fréquents :")
display(
    pd.Series(events, name="nombre_evenements")
    .sort_values(ascending=False)
    .head(15)
    .to_frame()
)

print("\nRépartition par site :")
display(
    pd.Series(sites, name="nombre_evenements")
    .sort_values(ascending=False)
    .to_frame()
)

print("\nRépartition par ligne :")
display(
    pd.Series(production_lines, name="nombre_evenements")
    .sort_index()
    .to_frame()
)

print("\nExemples de lignes non reconnues :")
if malformed_examples:
    for example in malformed_examples:
        print("-", example)
else:
    print("Aucune ligne malformée détectée avec le motif utilisé.")

print(
    "\nNombre de operator_login :",
    sum("operator_login" in line for line in log_lines),
)
print(
    "Nombre de vibration_overlimit :",
    sum("vibration_overlimit" in line for line in log_lines),
)
print(
    "Nombre de maintenance_completed :",
    sum("maintenance_completed" in line for line in log_lines),
)


Période observée : 2026-04-01T00:00:16 → 2026-04-29T23:57:41
Lignes reconnues : 30000
Lignes non reconnues : 0

Niveaux de logs :


,nombre_evenements
INFO,22501
WARN,5758
ERROR,1741



Événements les plus fréquents :


,nombre_evenements
operator_login,4615
maintenance_completed,4501
machine_started,4488
shift_changed,4473
tooling_loaded,4424
vibration_threshold_approached,1443
temperature_drift_detected,1442
lubricant_low,1440
throughput_below_target,1433
communication_lost,468



Répartition par site :


,nombre_evenements
Saint-Etienne,12041
Roubaix,11970
Lyon,5989



Répartition par ligne :


,nombre_evenements
LINE-1,13012
LINE-2,7021
LINE-3,6945
LINE-4,3022



Exemples de lignes non reconnues :
Aucune ligne malformée détectée avec le motif utilisé.

Nombre de operator_login : 4615
Nombre de vibration_overlimit : 414
Nombre de maintenance_completed : 4501


### Observations — Logs machines

- **Volume :** 30 000 lignes pour environ 1,8 Mo.
- **Période :** la période exacte est affichée dans la cellule précédente.
- **Format :** texte brut, avec un événement horodaté par ligne.
- **Qualité observée :**
  - la structure générale comprend un horodatage, un site, une ligne, un niveau et un message ;
  - certains messages contiennent un identifiant de capteur ou d'autres paramètres ;
  - les éventuelles lignes non reconnues sont affichées sans être corrigées ;
  - une documentation des codes événementiels sera nécessaire pour une ingestion fiable.
- **Risque RGPD :**
  - des événements comme `operator_login` peuvent permettre de suivre l'activité d'un opérateur ;
  - croisés avec `ouvrier_id`, le site, la ligne et l'heure, les logs peuvent contribuer à réidentifier un salarié.
- **Pertinence métier :**
  - les logs peuvent expliquer le contexte précédant une anomalie ;
  - ils peuvent signaler un dépassement, un changement d'équipe, un chargement d'outil ou une maintenance ;
  - ils sont utiles, mais plus complexes à intégrer que les sources tabulaires.
- **Questions pour Sébastien :**
  1. Existe-t-il une documentation officielle des événements machines ?
  2. Les niveaux de logs sont-ils normalisés sur tous les sites ?
  3. `operator_login` contient-il un identifiant personnel dans la version complète des logs ?


## Risque de réidentification par croisement

Un risque important n'apparaît pas dans une seule colonne, mais dans la combinaison suivante :

```text
ERP.ouvrier_id
+ logs.operator_login
+ site
+ line_id
+ timestamp
= possibilité de reconstruire l'activité d'un salarié
```

### Recommandation initiale

- Vérifier si `ouvrier_id` est réellement nécessaire au modèle.
- Le supprimer ou le pseudonymiser avant ingestion lorsque l'identité n'est pas utile.
- Limiter les accès aux données détaillées.
- Définir une durée de conservation.
- Faire valider la finalité du traitement par le DPO.


## Synthèse des sources

Le tableau suivant fournit une première cartographie. Les résultats détaillés restent dans les cellules précédentes.


In [25]:
inventory = pd.DataFrame(
    [
        {
            "source": "Capteurs IoT",
            "format": "CSV",
            "volume": f"{len(df_iot):,} lignes",
            "frequence": "Une mesure/seconde annoncée",
            "qualite_principale": "Vibrations partiellement manquantes",
            "risque_rgpd": "Réidentification indirecte par croisement",
            "priorite": "Haute",
        },
        {
            "source": "ERP",
            "format": "JSON",
            "volume": f"{len(df_erp):,} ordres",
            "frequence": "Export quotidien",
            "qualite_principale": "ouvrier_id partiellement manquant",
            "risque_rgpd": "Identifiant salarié indirect",
            "priorite": "Haute",
        },
        {
            "source": "Logs machines",
            "format": "Texte brut",
            "volume": f"{len(log_lines):,} événements",
            "frequence": "Événementielle",
            "qualite_principale": "Format semi-structuré à documenter",
            "risque_rgpd": "Suivi potentiel de l'activité",
            "priorite": "Moyenne à haute",
        },
    ]
)

display(inventory)


,source,format,volume,frequence,qualite_principale,risque_rgpd,priorite
0,Capteurs IoT,CSV,"51,000 lignes",Une mesure/seconde annoncée,Vibrations partiellement manquantes,Réidentification indirecte par croisement,Haute
1,ERP,JSON,"2,000 ordres",Export quotidien,ouvrier_id partiellement manquant,Identifiant salarié indirect,Haute
2,Logs machines,Texte brut,"30,000 événements",Événementielle,Format semi-structuré à documenter,Suivi potentiel de l'activité,Moyenne à haute


## Conclusion pour `identification_sources.md`

### Sources à considérer en priorité

1. **Capteurs IoT** : source principale pour observer les dérives physiques.
2. **ERP** : source nécessaire pour apporter le contexte de fabrication.
3. **Logs machines** : source explicative utile, mais nécessitant une structuration et une documentation supplémentaires.

### Points restant à clarifier

1. Quelle est la clé fiable permettant de relier les mesures IoT, les ordres ERP, les logs et les non-conformités ?
2. Quel est le niveau de performance actuel du modèle, notamment son rappel et sa précision ?
3. L'objectif annoncé correspond-il à une réduction de 20 % des non-conformités ou à un taux final de 20 % ?
4. Quel est le coût exact d'une pièce non conforme ?
5. Quel temps d'inférence maximal est acceptable ?
6. Quelles données personnelles sont réellement nécessaires au modèle ?
7. Quelle infrastructure héberge actuellement le modèle et quelles sont ses limites ?

> Après exécution complète du notebook, reporter les résultats utiles dans `../identification_sources.md`.
